# Plotly — Interactive Web-Ready Visualizations

## What is Plotly?

Plotly is a **visualization library that creates interactive charts** — charts you can hover over, zoom into, click on, and share on the web. While Matplotlib and Seaborn produce static images (like photos), Plotly produces interactive experiences (like web apps).

**Real-world analogy**: Matplotlib is a printed map — beautiful and informative, but static. Plotly is Google Maps — you can zoom in, click on locations, get details on hover, and explore dynamically. Both are useful, but for different purposes.

## Why Plotly?

| Use Case | Tool |
|----------|------|
| Static publication/PDF figures | Matplotlib / Seaborn |
| Quick EDA in Jupyter | Seaborn |
| **Interactive dashboards for stakeholders** | **Plotly** |
| **Web apps (with Dash)** | **Plotly** |
| Presentations with zoom/hover | Plotly |
| Geographic/map visualizations | Plotly |

## Two APIs in Plotly

1. **Plotly Express (`px`)** — High-level, one-liner API. Like Seaborn for Plotly.
2. **Plotly Graph Objects (`go`)** — Low-level, full control. Like Matplotlib for Plotly.

**Start with `px`, drop to `go` when you need more control.**

## Prerequisites
- Python basics
- Pandas DataFrames

## Table of Contents
1. Installation & Setup
2. Plotly Express: Quick Interactive Charts
3. Core Chart Types (scatter, line, bar, histogram, box, violin)
4. Plotly Graph Objects: Full Control
5. Subplots with `make_subplots`
6. Geographic Maps
7. 3D Visualizations
8. Layout Customization (`update_layout`, `update_traces`)
9. Animations
10. Exporting (HTML, PNG, PDF)
11. Common Pitfalls
12. Mini Project: Interactive Sales Analytics Dashboard
13. Interview Q&A
14. Resources

---

**Official Docs**: https://plotly.com/python/  
**Plotly Express**: https://plotly.com/python/plotly-express/  
**Gallery**: https://plotly.com/python/basic-charts/  
**YouTube (Charming Data)**: https://www.youtube.com/c/CharmingData (best Plotly tutorials!)  
**Dash for Dashboards**: https://dash.plotly.com/

## 1. Installation & Setup

```bash
pip install plotly pandas numpy
# For static image export:
pip install kaleido
```

In Jupyter, Plotly renders inline automatically. The charts are HTML+JavaScript — interactive in notebooks and browsers.

In [ ]:
import plotly.express as px          # High-level (like Seaborn)
import plotly.graph_objects as go    # Low-level (like Matplotlib)
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

import plotly
print(f"Plotly version: {plotly.__version__}")

# Plotly comes with several built-in datasets
print("\nBuilt-in datasets:", dir(px.data))

---
## 2. Plotly Express: Quick Interactive Charts

`px` uses the **same API pattern as Seaborn**: `px.chart_type(data=df, x='col', y='col', color='cat_col')`.

But the result is **interactive** — hover for details, click legend to toggle, double-click to zoom.

> **Interactive features built-in** (no extra code needed):
> - Hover tooltips with data values
> - Zoom (scroll or drag)
> - Pan (drag after zooming)
> - Click legend to show/hide traces
> - Download button (PNG)
> - Reset zoom (double-click)

In [ ]:
# Load the classic Gapminder dataset: country statistics over time
gapminder = px.data.gapminder()
print("Gapminder shape:", gapminder.shape)
print(gapminder.head())

# ── Scatter Plot: Life Expectancy vs GDP Per Capita ───────────────────────────
# Each point = one country; size = population; color = continent
fig = px.scatter(
    gapminder[gapminder.year == 2007],  # Just year 2007
    x='gdpPercap',
    y='lifeExp',
    size='pop',               # Bubble size = population
    color='continent',        # Color by continent
    hover_name='country',     # Shows country name on hover
    log_x=True,               # Log scale on x-axis
    size_max=60,              # Max bubble size
    title='Life Expectancy vs GDP per Capita (2007)',
    labels={'gdpPercap': 'GDP per Capita (USD, log scale)',
            'lifeExp': 'Life Expectancy (years)',
            'pop': 'Population'},
    template='plotly_white'   # Clean white background
)
fig.show()
print("▲ Hover over bubbles to see country details! Click continent legend to filter.")

---
## 3. Core Chart Types with Plotly Express

In [ ]:
# ── Line Chart: Time series ───────────────────────────────────────────────────
# GDP over time for top 5 most populous countries
top_countries = ['China', 'India', 'United States', 'Indonesia', 'Brazil']
fig_line = px.line(
    gapminder[gapminder.country.isin(top_countries)],
    x='year', y='gdpPercap', color='country',
    title='GDP per Capita over Time: Top 5 Countries',
    markers=True,  # Show data points
    labels={'gdpPercap': 'GDP per Capita ($)', 'year': 'Year'},
    template='plotly_white'
)
fig_line.show()

# ── Bar Chart: Average life expectancy by continent ────────────────────────────
gap_2007 = gapminder[gapminder.year == 2007].copy()
continent_avg = gap_2007.groupby('continent').agg(
    avg_life=('lifeExp', 'mean'),
    n_countries=('country', 'count')
).reset_index().sort_values('avg_life', ascending=False)

fig_bar = px.bar(
    continent_avg,
    x='continent', y='avg_life',
    color='continent',
    text='avg_life',  # Show value on bar
    title='Average Life Expectancy by Continent (2007)',
    labels={'avg_life': 'Avg Life Expectancy (years)'},
    template='plotly_white'
)
fig_bar.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig_bar.show()

In [ ]:
np.random.seed(42)
tips = px.data.tips()

# ── Histogram: interactive with hover ─────────────────────────────────────────
fig_hist = px.histogram(
    tips, x='total_bill', color='sex',
    nbins=25, barmode='overlay',  # 'overlay', 'stack', 'group'
    marginal='box',               # Show a box plot on the margin!
    opacity=0.7,
    title='Total Bill Distribution by Gender',
    template='plotly_white'
)
fig_hist.show()
print("▲ The marginal='box' adds a box plot summary above the histogram!")

# ── Box Plot ───────────────────────────────────────────────────────────────────
fig_box = px.box(
    tips, x='day', y='total_bill', color='smoker',
    notched=True,             # Notch shows CI around median
    points='outliers',        # Show outlier points; 'all' shows all points
    category_orders={'day': ['Thur', 'Fri', 'Sat', 'Sun']},
    title='Total Bill by Day and Smoking Status',
    template='plotly_white'
)
fig_box.show()

# ── Violin + Scatter combined ─────────────────────────────────────────────────
fig_violin = px.violin(
    tips, x='day', y='tip', color='sex',
    box=True,            # Show box plot inside violin
    points='all',        # Show all data points
    hover_data=tips.columns,
    category_orders={'day': ['Thur', 'Fri', 'Sat', 'Sun']},
    title='Tip Distribution: Violin + Box + Points',
    template='plotly_white'
)
fig_violin.show()

In [ ]:
# ── Sunburst Chart: Hierarchical data ─────────────────────────────────────────
# Great for part-of-whole with hierarchy: continent → country
fig_sun = px.sunburst(
    gap_2007,
    path=['continent', 'country'],  # Hierarchy: click to drill down!
    values='pop',                   # Slice size = population
    color='lifeExp',               # Color = life expectancy
    color_continuous_scale='RdYlGn',
    title='World Population by Continent and Country (2007)'
)
fig_sun.show()
print("▲ Click on a continent to zoom in! Click center to zoom out.")

# ── Treemap: Alternative to Sunburst ─────────────────────────────────────────
fig_tree = px.treemap(
    gap_2007[gap_2007.continent == 'Americas'],
    path=['continent', 'country'],
    values='pop',
    color='gdpPercap',
    color_continuous_scale='Blues',
    title='Americas: Population (size) and GDP (color)'
)
fig_tree.show()

# ── Heatmap with px ────────────────────────────────────────────────────────────
# Pivot: life expectancy by continent over years
pivot = gapminder.pivot_table(values='lifeExp', index='continent', columns='year')
fig_heat = px.imshow(
    pivot,
    color_continuous_scale='YlOrRd',
    title='Life Expectancy: Continent × Year',
    labels={'x': 'Year', 'y': 'Continent', 'color': 'Life Exp'},
    text_auto='.1f'  # Show values in cells
)
fig_heat.show()

---
## 4. Plotly Graph Objects: Full Control

`go` gives you precise control over every visual element. The structure is:

```python
fig = go.Figure()
fig.add_trace(go.ChartType(x=..., y=..., name=...))  # Add data layers
fig.update_layout(title=..., xaxis_title=...)         # Customize appearance
fig.show()
```

**When to use `go`**: When you need to mix chart types (e.g., bar + line), add shapes/annotations, or control specific visual properties that `px` doesn't expose.

In [ ]:
np.random.seed(42)
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
revenue = [185, 192, 205, 198, 220, 235, 285, 260, 248, 265, 290, 320]
costs   = [150, 155, 160, 162, 170, 178, 195, 185, 180, 188, 200, 215]
profit  = [r - c for r, c in zip(revenue, costs)]

# ── Combo chart: bars + line ───────────────────────────────────────────────────
fig = go.Figure()

# Bar traces for Revenue and Costs
fig.add_trace(go.Bar(
    x=months, y=revenue,
    name='Revenue',
    marker_color='#3498db',
    opacity=0.8
))

fig.add_trace(go.Bar(
    x=months, y=costs,
    name='Costs',
    marker_color='#e74c3c',
    opacity=0.8
))

# Line trace for Profit on secondary y-axis
fig.add_trace(go.Scatter(
    x=months, y=profit,
    name='Profit',
    mode='lines+markers',
    marker=dict(size=8, color='#2ecc71'),
    line=dict(width=3, color='#2ecc71'),
    yaxis='y2'  # Plot on secondary y-axis
))

# Customize layout
fig.update_layout(
    title=dict(text='Monthly Financials: Revenue, Costs & Profit', font_size=16),
    barmode='group',  # 'group' (side by side) or 'stack'
    xaxis=dict(title='Month'),
    yaxis=dict(title='Amount ($K)', side='left'),
    yaxis2=dict(
        title='Profit ($K)',
        overlaying='y',  # Share the same x-axis
        side='right',
        showgrid=False
    ),
    legend=dict(x=0.01, y=0.99),
    template='plotly_white',
    hovermode='x unified'  # Shows all traces at the same x on hover
)

# Add an annotation
fig.add_annotation(
    x='Jul', y=285,
    text='Summer Sale!',
    showarrow=True,
    arrowhead=2,
    bgcolor='lightyellow',
    bordercolor='orange'
)

# Add a horizontal reference line
fig.add_hline(y=250, line_dash='dash', line_color='gray', 
               annotation_text='Revenue Target: $250K')

fig.show()

---
## 5. Subplots with `make_subplots`

In [ ]:
from plotly.subplots import make_subplots

# Create a 2×2 subplot layout
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Revenue by Month', 'Cost Breakdown',
        'Profit Margin Trend', 'Revenue vs Cost'
    ),
    specs=[
        [{}, {}],              # Row 1: standard charts
        [{}, {'type': 'scatter'}]  # Row 2: standard charts
    ]
)

# Panel 1: Line chart
fig.add_trace(
    go.Scatter(x=months, y=revenue, mode='lines+markers', 
               name='Revenue', line=dict(color='#3498db')),
    row=1, col=1
)

# Panel 2: Bar chart
cost_breakdown = {'Salaries': 120, 'Rent': 30, 'Marketing': 25, 'Other': 20}
fig.add_trace(
    go.Bar(x=list(cost_breakdown.keys()), y=list(cost_breakdown.values()),
           name='Cost Breakdown', marker_color=['#e74c3c','#e67e22','#f39c12','#95a5a6']),
    row=1, col=2
)

# Panel 3: Area chart for profit margin
margin = [p/r*100 for p, r in zip(profit, revenue)]
fig.add_trace(
    go.Scatter(x=months, y=margin, fill='tozeroy', 
               name='Profit Margin %', line=dict(color='#2ecc71')),
    row=2, col=1
)

# Panel 4: Scatter
fig.add_trace(
    go.Scatter(x=costs, y=revenue, mode='markers+text',
               text=months, textposition='top center',
               name='Rev vs Cost', marker=dict(color=profit, colorscale='RdYlGn', size=10)),
    row=2, col=2
)

fig.update_layout(
    title_text='Financial Overview Dashboard',
    height=600,
    showlegend=True,
    template='plotly_white'
)
fig.show()

---
## 6. Geographic Maps

Plotly makes geographic visualizations straightforward with `px.choropleth` (filled maps) and `px.scatter_geo` (point maps). This is a major advantage over Matplotlib/Seaborn.

In [ ]:
# ── Choropleth: World map colored by GDP per capita ────────────────────────────
fig_map = px.choropleth(
    gap_2007,
    locations='iso_alpha',      # 3-letter ISO country codes
    color='gdpPercap',          # Color by this value
    hover_name='country',       # Country name on hover
    hover_data={'pop': ':,.0f', 'lifeExp': ':.1f'},
    color_continuous_scale='Viridis',
    title='World GDP per Capita (2007)',
    projection='natural earth'   # Map projection style
)
fig_map.update_layout(coloraxis_colorbar=dict(title='GDP/Capita'))
fig_map.show()
print("▲ Hover over countries for details! Zoom and pan freely.")

# ── Bubble Map: US cities with population bubbles ─────────────────────────────
us_cities = pd.DataFrame({
    'City':  ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix',
              'Philadelphia', 'San Antonio', 'San Diego', 'Dallas', 'San Jose'],
    'Lat':   [40.7128, 34.0522, 41.8781, 29.7604, 33.4484,
              39.9526, 29.4241, 32.7157, 32.7767, 37.3382],
    'Lon':   [-74.0060, -118.2437, -87.6298, -95.3698, -112.0740,
              -75.1652, -98.4936, -117.1611, -96.7970, -121.8863],
    'Pop':   [8336817, 3979576, 2693976, 2304580, 1608139,
              1584064, 1547253, 1423851, 1343573, 1021795],
    'Revenue': [2800, 1900, 1400, 1200, 980, 850, 780, 720, 1100, 1500]  # M$
})

fig_bubble = px.scatter_geo(
    us_cities,
    lat='Lat', lon='Lon',
    size='Pop',
    color='Revenue',
    hover_name='City',
    size_max=40,
    color_continuous_scale='Blues',
    scope='usa',
    title='US Cities: Population (size) & Revenue (color)'
)
fig_bubble.update_layout(geo=dict(showland=True, landcolor='lightgray'))
fig_bubble.show()

---
## 7. 3D Visualizations

Plotly's 3D charts are interactive — you can rotate, zoom, and orbit around them by clicking and dragging. Static libraries can't do this!

In [ ]:
# ── 3D Scatter Plot ───────────────────────────────────────────────────────────
iris = px.data.iris()
fig_3d = px.scatter_3d(
    iris,
    x='sepal_length', y='sepal_width', z='petal_length',
    color='species',
    size='petal_width',
    opacity=0.7,
    title='Iris Dataset: 3D Scatter (drag to rotate!)',
    template='plotly_white'
)
fig_3d.show()
print("▲ Click and drag to rotate the 3D scatter plot!")

# ── 3D Surface Plot ───────────────────────────────────────────────────────────
x_surf = np.linspace(-3, 3, 50)
y_surf = np.linspace(-3, 3, 50)
X, Y   = np.meshgrid(x_surf, y_surf)
Z      = np.sin(np.sqrt(X**2 + Y**2))  # sinc-like function

fig_surf = go.Figure(data=[go.Surface(
    z=Z, x=x_surf, y=y_surf,
    colorscale='Viridis',
    showscale=True
)])
fig_surf.update_layout(
    title='3D Surface: sin(√(x²+y²)) — Drag to Rotate',
    scene=dict(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
    ),
    width=700, height=500
)
fig_surf.show()

---
## 8. Animations

Plotly can animate charts over time with a play button — perfect for showing change across years or steps. This is one of Plotly's killer features.

In [ ]:
# ── Animated Bubble Chart: Gapminder over 50 years ────────────────────────────
# This is literally the famous Hans Rosling visualization!
fig_anim = px.scatter(
    gapminder,
    x='gdpPercap',
    y='lifeExp',
    animation_frame='year',      # This creates the animation slider!
    animation_group='country',   # Keeps each country's bubble continuous
    size='pop',
    color='continent',
    hover_name='country',
    log_x=True,
    size_max=55,
    range_x=[100, 100000],
    range_y=[25, 90],
    title='World Development 1952–2007 (Hit Play!)',
    labels={'gdpPercap': 'GDP per Capita', 'lifeExp': 'Life Expectancy', 'pop': 'Population'},
    template='plotly_white'
)
fig_anim.update_layout(height=550)
fig_anim.show()
print("▲ Press the PLAY button! Watch how the world's health & wealth evolved from 1952 to 2007.")
print("This is the famous Hans Rosling 'Our World in Data' visualization!")
print("YouTube: https://www.youtube.com/watch?v=jbkSRLYSojo")

---
## 9. Layout Customization

Two key methods for customization:
- `fig.update_layout()` — changes the overall figure (title, background, fonts, margins)
- `fig.update_traces()` — changes all traces (data series) properties at once

In [ ]:
# Create a base chart
fig = px.bar(tips, x='day', y='total_bill', color='time',
              category_orders={'day': ['Thur', 'Fri', 'Sat', 'Sun']},
              barmode='group', template='none')  # 'none' = no styling

# ── update_layout: full layout customization ──────────────────────────────────
fig.update_layout(
    title=dict(
        text='Restaurant Bills: Fully Customized Layout',
        x=0.5,              # Center the title
        font=dict(size=18, color='#2c3e50', family='Arial')
    ),
    xaxis=dict(
        title='Day of Week',
        tickfont=dict(size=13)
    ),
    yaxis=dict(
        title='Total Bill ($)',
        gridcolor='#ECF0F1',
        zeroline=True
    ),
    plot_bgcolor='#FAFAFA',   # Background of the plot area
    paper_bgcolor='white',    # Background outside the plot area
    legend=dict(
        title='Meal Time',
        orientation='h',      # Horizontal legend
        yanchor='bottom', y=1.02,  # Above the chart
        xanchor='right', x=1
    ),
    font=dict(family='Arial'),
    margin=dict(l=60, r=20, t=100, b=60),
    height=450
)

# ── update_traces: batch-modify all data series ───────────────────────────────
fig.update_traces(
    marker_line_color='white',
    marker_line_width=1.5,
    opacity=0.85
)

# ── Hover template customization ──────────────────────────────────────────────
fig.update_traces(
    hovertemplate='<b>%{x}</b><br>Bill: $%{y:.2f}<extra></extra>'
)

fig.show()
print("Key layout properties:")
print("  template:      'plotly_white', 'plotly_dark', 'ggplot2', 'seaborn', 'simple_white'")
print("  hovermode:     'x unified', 'closest', False")
print("  dragmode:      'zoom', 'pan', 'select', 'lasso'")

---
## 10. Exporting Plotly Figures

| Format | Code | Notes |
|--------|------|-------|
| **Interactive HTML** | `fig.write_html('chart.html')` | Share with anyone; open in browser |
| **Static PNG** | `fig.write_image('chart.png')` | Requires `kaleido` |
| **Static PDF** | `fig.write_image('chart.pdf')` | Requires `kaleido` |
| **Static SVG** | `fig.write_image('chart.svg')` | Vector, requires `kaleido` |
| **JSON** | `fig.write_json('chart.json')` | For saving and reloading |

```bash
pip install kaleido  # Required for static image export
```

In [ ]:
import os

# Create a simple figure to export
fig_export = px.scatter(iris, x='sepal_length', y='sepal_width', 
                         color='species', title='Iris Export Example')

# ── Export as interactive HTML (no extra deps) ─────────────────────────────────
fig_export.write_html('/tmp/iris_chart.html')
print(f"HTML saved: {os.path.getsize('/tmp/iris_chart.html'):,} bytes")
print("Open iris_chart.html in a browser — fully interactive!")

# ── Export as JSON (can reload later) ─────────────────────────────────────────
fig_export.write_json('/tmp/iris_chart.json')
print(f"JSON saved: {os.path.getsize('/tmp/iris_chart.json'):,} bytes")

# ── Reload from JSON ───────────────────────────────────────────────────────────
import plotly.io as pio
fig_reloaded = pio.read_json('/tmp/iris_chart.json')
print(f"Reloaded figure title: {fig_reloaded.layout.title.text}")

# ── Static export (requires kaleido) ──────────────────────────────────────────
try:
    fig_export.write_image('/tmp/iris_chart.png', width=800, height=500, scale=2)
    print(f"PNG saved: {os.path.getsize('/tmp/iris_chart.png'):,} bytes")
except Exception as e:
    print(f"Static export needs kaleido: pip install kaleido (Error: {e})")

---
## 11. Common Pitfalls

| Pitfall | Problem | Fix |
|---------|---------|-----|
| `fig.show()` shows nothing in script | Plotly needs a browser/notebook | Use `fig.write_html()` and open in browser; or `fig.show(renderer='browser')` |
| Static export gives blank image | `kaleido` not installed | `pip install kaleido` |
| Too many data points → slow | 100K+ points makes interactive hover slow | Downsample or use `px.density_heatmap` instead of scatter |
| Mixing `px` and `go` figures | Sometimes traces conflict | Keep one style per figure; use `go.Figure(px.scatter(...))` to convert |
| `update_traces(selector=...)` needed | `update_traces` applies to ALL traces | Use `selector=dict(type='bar')` to target only bar traces |
| Categorical order is wrong | Plotly infers order from data | Use `category_orders={'col': ['A','B','C']}` to set order explicitly |
| Legend overlaps chart | Default legend position inside plot | `legend=dict(x=1.02, y=1, xanchor='left')` to put outside |

---
## 12. Mini Project: Interactive Sales Analytics Dashboard

**Scenario**: You're building an analytics dashboard for a SaaS company. The dashboard needs to show: MRR trend, customer breakdown by tier, churn funnel, and geographic distribution — all interactive.

**Approach**: Use `make_subplots` with `go` traces to build a multi-panel interactive dashboard in a single figure.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

np.random.seed(42)

# ── Generate SaaS Business Data ───────────────────────────────────────────────
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# MRR (Monthly Recurring Revenue)
mrr        = [45, 52, 58, 63, 72, 80, 95, 105, 112, 128, 142, 165]  # K$
new_mrr    = [8, 10, 9, 11, 14, 12, 20, 15, 14, 20, 18, 28]         # K$ from new customers
churned_mrr= [3, 4, 3, 4, 5, 4, 5, 5, 4, 4, 4, 5]                   # K$ lost
expansion  = [2, 3, 3, 4, 5, 6, 8, 5, 6, 8, 7, 10]                  # K$ from upsells

# Customer tiers
tiers      = ['Free', 'Starter', 'Professional', 'Enterprise']
customers  = [1200, 450, 180, 45]
mrr_by_tier= [0, 45, 108, 225]  # revenue contribution

# Cohort retention (% of customers retained after N months)
cohort_months = list(range(1, 13))
retention_2022 = [100, 82, 71, 65, 61, 58, 56, 55, 54, 53, 52, 51]
retention_2023 = [100, 85, 76, 71, 68, 65, 63, 62, 61, 60, 59, 58]

# ── Build Dashboard ───────────────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'MRR Growth & Components',
        'Customer Mix by Tier',
        'Revenue by Tier',
        'MRR Movement (Waterfall)',
        'Cohort Retention',
        'Net Revenue Retention'
    ],
    specs=[
        [{}, {'type': 'pie'}, {'type': 'bar'}],
        [{}, {}, {}]
    ],
    horizontal_spacing=0.1,
    vertical_spacing=0.15
)

# Panel 1: MRR components (stacked area)
fig.add_trace(go.Scatter(x=months, y=mrr, name='Total MRR', 
                          mode='lines+markers', line=dict(color='#3498db', width=3)), row=1, col=1)
fig.add_trace(go.Bar(x=months, y=new_mrr, name='New MRR', 
                      marker_color='#2ecc71', opacity=0.7), row=1, col=1)
fig.add_trace(go.Bar(x=months, y=expansion, name='Expansion', 
                      marker_color='#f39c12', opacity=0.7), row=1, col=1)
fig.add_trace(go.Bar(x=months, y=[-c for c in churned_mrr], name='Churned MRR', 
                      marker_color='#e74c3c', opacity=0.7), row=1, col=1)

# Panel 2: Pie chart — customer mix
fig.add_trace(go.Pie(
    labels=tiers, values=customers,
    name='Customers', hole=0.4,  # donut chart
    marker=dict(colors=['#95a5a6','#3498db','#e67e22','#2ecc71'])
), row=1, col=2)

# Panel 3: Revenue by tier
fig.add_trace(go.Bar(
    x=tiers, y=mrr_by_tier,
    name='MRR by Tier',
    marker_color=['#95a5a6','#3498db','#e67e22','#2ecc71'],
    text=[f'${v}K' for v in mrr_by_tier],
    textposition='outside'
), row=1, col=3)

# Panel 4: Dec month waterfall decomposition
dec_waterfall = ['Start', 'New MRR', 'Expansion', 'Churn', 'End']
start_val = mrr[-2]  # Nov MRR
fig.add_trace(go.Waterfall(
    x=dec_waterfall,
    y=[start_val, new_mrr[-1], expansion[-1], -churned_mrr[-1], 0],
    measure=['absolute', 'relative', 'relative', 'relative', 'total'],
    name='Dec Waterfall',
    connector={'line': {'color': 'gray'}},
    increasing={'marker': {'color': '#2ecc71'}},
    decreasing={'marker': {'color': '#e74c3c'}},
    totals={'marker': {'color': '#3498db'}}
), row=2, col=1)

# Panel 5: Cohort retention
fig.add_trace(go.Scatter(x=cohort_months, y=retention_2022, name='2022 Cohort',
                          mode='lines+markers', line=dict(color='#e74c3c')), row=2, col=2)
fig.add_trace(go.Scatter(x=cohort_months, y=retention_2023, name='2023 Cohort',
                          mode='lines+markers', line=dict(color='#3498db')), row=2, col=2)

# Panel 6: NRR gauge
nrr = [(mrr[i] - churned_mrr[i] + expansion[i]) / mrr[max(0, i-1)] * 100 if i > 0 else 100 
       for i in range(12)]
fig.add_trace(go.Scatter(x=months, y=nrr, name='NRR %', fill='tozeroy',
                          line=dict(color='#9b59b6')), row=2, col=3)
fig.add_hline(y=100, line_dash='dash', line_color='red', row=2, col=3)

# ── Global Layout ─────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(text='SaaS Business Dashboard — FY 2024', font_size=20, x=0.5),
    height=700,
    showlegend=True,
    template='plotly_white',
    barmode='relative',
    legend=dict(orientation='h', yanchor='bottom', y=-0.15),
    hovermode='x unified'
)

# Update individual axis labels
fig.update_yaxes(title_text='MRR ($K)', row=1, col=1)
fig.update_yaxes(title_text='Retention %', row=2, col=2)
fig.update_yaxes(title_text='NRR %', row=2, col=3)
fig.update_xaxes(title_text='Month after Acquisition', row=2, col=2)

fig.write_html('/tmp/saas_dashboard.html')
print("Dashboard saved to /tmp/saas_dashboard.html")
fig.show()

---
## 13. Interview Q&A

**Q1: When would you use Plotly vs Matplotlib/Seaborn?**  
**A**: Use Plotly when: (1) you need interactivity (hover, zoom, click), (2) you're building a web dashboard or Dash app, (3) you're presenting to stakeholders who need to explore data, (4) you're making geographic or 3D visualizations. Use Matplotlib/Seaborn for: static figures in papers, when you need specific statistical plots (Seaborn's regplot, pairplot), or when file size matters (HTML files are larger than PNG).

---
**Q2: What is the difference between Plotly Express and Plotly Graph Objects?**  
**A**: Plotly Express (`px`) is a high-level API — one function call creates a complete chart. It's great for standard charts. Graph Objects (`go`) is the low-level API — you add traces manually for full control. `px` actually creates `go` objects internally. Use `px` for speed; use `go` when you need custom traces, mixed chart types, or specific properties `px` doesn't expose.

---
**Q3: How do you add a secondary y-axis in Plotly?**  
**A**: Add `yaxis='y2'` to the secondary trace, then in `update_layout` add `yaxis2=dict(overlaying='y', side='right')`. The `overlaying='y'` means it shares the same x-axis with the primary y-axis.

---
**Q4: How does Plotly handle animations?**  
**A**: In Plotly Express, add `animation_frame='column_name'` — Plotly creates one frame per unique value in that column and adds a play button and slider. In Graph Objects, use `go.Figure(frames=[go.Frame(data=..., name=str(val)) for val in values])` and add `updatemenus` with a Play button.

---
**Q5: What is a waterfall chart and when is it used?**  
**A**: A waterfall chart shows how a starting value changes through a series of positive and negative contributions to reach a final value. Common uses: showing how revenue breaks down into profit (revenue → COGS → OpEx → profit), or how MRR changes month-over-month (start → new → expansion → churn → end).

---
**Q6: How do you embed a Plotly chart in a web page?**  
**A**: `fig.write_html('chart.html')` creates a self-contained HTML file with the chart embedded as JavaScript. This file can be opened in any browser or embedded in a web page with an `<iframe>`. For production web apps, use the Dash framework (by Plotly) which creates full interactive web applications with Python callbacks.

---
## 14. Resources

### Official
- **Plotly Python Docs**: https://plotly.com/python/
- **Plotly Express Reference**: https://plotly.com/python-api-reference/plotly.express.html
- **Graph Objects Reference**: https://plotly.com/python-api-reference/plotly.graph_objects.html
- **Dash (build web apps)**: https://dash.plotly.com/

### YouTube
- **Charming Data (Adam Schroeder)** — best Plotly channel: https://www.youtube.com/c/CharmingData
  - Recommended: "Plotly Express in 10 min", "Dash Tutorial Series"
- **Python Engineer — Plotly Tutorial**: https://www.youtube.com/watch?v=GGL6U0k8WYA
- **Keith Galli — Plotly**: https://www.youtube.com/watch?v=_b2KXL0wHQg

### Articles
- **Plotly Blog**: https://medium.com/plotly
- **Towards Data Science — Plotly**: https://towardsdatascience.com/tagged/plotly

---
## Summary & What's Next

### What You Learned
| Concept | Key Point |
|---------|----------|
| px vs go | Start with `px` (fast); drop to `go` (full control) |
| Core charts | scatter, line, bar, histogram, box, violin, sunburst, treemap |
| Maps | `choropleth` (filled), `scatter_geo` (bubbles) |
| 3D | `scatter_3d`, `Surface` — all interactive (drag to rotate) |
| Animations | `animation_frame='year'` → play button |
| Subplots | `make_subplots(rows, cols)` + `fig.add_trace(..., row, col)` |
| Layout | `update_layout()` for figure; `update_traces()` for data |
| Export | `write_html()` (interactive), `write_image()` (static, needs kaleido) |

### What's Next?
- **Next Notebook**: Bokeh — another interactive library, better for streaming data and Python callbacks
- **Explore**: Build a Dash app: `pip install dash` and extend the dashboard with dropdown filters
- **Challenge**: Recreate a Plotly chart from the gallery: https://plotly.com/python/

> **Key insight**: Plotly shines for stakeholder-facing dashboards and exploratory presentations. When your audience needs to interact with data — not just view it — Plotly is the right tool. Combine it with Dash for full web applications.